# basic

In [1]:
%load_ext autoreload
%autoreload all

In [2]:
import polars as pl
import pickle
import numpy as np


import src.configs as configs
from src.tokenizer import GraphTokenizer


# 0. load files and tokenizer !!! i prefiltered the non-sense candidates rows

In [3]:
with open(configs.ProcessedGraph().candidate_is_a_reachable_dict, "rb") as f:
    candidate_reachable_child_map = pickle.load(f)

df_concept_hug_w_depth = pl.read_parquet(configs.ProcessedGraph.concept_w_depth)
mapped_candidate_rel_dist_prop = (pl.read_parquet(configs.ProcessedGraph().mapped_candidate_rel_dist_prop)
                                  
                                  # remove all relations in subgraph where candidate and mapped are distant
                                  .filter(pl.col("distance") <= configs.TokenizerParam().max_dist_candidate)
                                  )

In [4]:
tokenizer = GraphTokenizer(df_concept_hug_w_depth,
                        mapped_candidate_rel_dist_prop,
                        candidate_reachable_child_map,
                        configs.TokenizerParam().max_dist_candidate
                        )

Ks = np.arange(500, 12000, 500)

# all selections candidate generation

## 1. all mapped are candidates

In [5]:
candidate_ids = tokenizer.mapped_id_list
scores, _, _ = tokenizer.evaluate_components_and_tokenize(candidate_ids)
scores

{'sem_cov_score': 1.0,
 'distance_score': 1.0,
 'uniqueness_entropy_score': 0.9999999999999999,
 'conciseness_score': 1.0,
 'compression_rate': 1.0,
 'UNK_rate': 0.0,
 'exact_rate': 1.0}

## 2. random K

In [6]:
n_samples = 50
candidates_all = pl.Series(tokenizer.candidate_concepts)

samples = [
    pl.DataFrame({
        "k": k,
        "iter": s,
        "candidate_id": candidates_all.sample(n=k),
    })
    for k in Ks
    for s in range(n_samples)
]

pl.concat(samples).write_parquet(f"{configs.Baselines().path}k_random_all_samples.parquet")

## 3. highest degree ancester (general candidates removed)

In [14]:
(mapped_candidate_rel_dist_prop
 .group_by("dst.id")
 .agg(num_cpts = pl.col("src.id").n_unique())
 .sort("num_cpts", descending = True)
 .with_row_index()
).write_parquet(f"{configs.Baselines().path}highest_degree.parquet")

## 4. highest degree ancester with distance == 1 (general candidates removed)

In [16]:
(mapped_candidate_rel_dist_prop
 .filter(pl.col("distance") == 1)
 .group_by("dst.id")
 .agg(num_cpts = pl.col("src.id").n_unique())
 .sort("num_cpts", descending = True)
 .with_row_index()
).write_parquet(f"{configs.Baselines().path}highest_degree_dist_1.parquet")

## 5. most children (general candidates not removed)

In [20]:
row = []
for candidate in candidate_reachable_child_map:
    row.append({
        "candidate": candidate,
        "num_reachable_child": len(candidate_reachable_child_map[candidate])
    })
(pl.DataFrame(row)
 .sort("num_reachable_child", descending=True)
 .with_row_index()
 ).write_parquet(f"{configs.Baselines().path}most_children.parquet")
